# 00 Setup - create every pipeline table

Safe to re-run (`CREATE TABLE IF NOT EXISTS`). Set `reset_silver = yes` to drop and rebuild
ONLY the Silver table and Silver quarantine (needed once, because Silver gained new columns).
Bronze, Bronze quarantine and the execution log are never dropped.

In [0]:
dbutils.widgets.dropdown("reset_silver", "no", ["no", "yes"])
reset_silver = dbutils.widgets.get("reset_silver")

if reset_silver == "yes":
    spark.sql("DROP TABLE IF EXISTS workspace.default.silver_market_observation")
    spark.sql("DROP TABLE IF EXISTS workspace.default.silver_quarantine")
    print("Dropped Silver table and Silver quarantine")

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.default.bronze_coin_market_raw (
    coin_id STRING NOT NULL,
    source_file STRING,
    timestamp_ms BIGINT,
    price_usd DOUBLE,
    market_cap_usd DOUBLE,
    total_volume_usd DOUBLE,
    load_timestamp TIMESTAMP NOT NULL,
    observation_timestamp TIMESTAMP
) USING DELTA
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.default.bronze_quarantine (
    coin_id STRING,
    source_file STRING,
    timestamp_ms BIGINT,
    observation_timestamp TIMESTAMP,
    price_usd DOUBLE,
    market_cap_usd DOUBLE,
    total_volume_usd DOUBLE,
    rejection_reason STRING,
    load_timestamp TIMESTAMP
) USING DELTA
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.default.bronze_file_quarantine (
    source_file STRING,
    processing_date STRING,
    load_type STRING,
    rejection_reason STRING,
    quarantine_timestamp TIMESTAMP
) USING DELTA
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.default.silver_market_observation (
    coin_id STRING NOT NULL,
    observation_timestamp TIMESTAMP NOT NULL,
    price_usd DOUBLE,
    market_cap_usd DOUBLE,
    total_volume_usd DOUBLE,
    source_file STRING,
    bronze_load_timestamp TIMESTAMP,
    load_timestamp TIMESTAMP NOT NULL,
    run_id STRING
) USING DELTA
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.default.silver_quarantine (
    coin_id STRING,
    source_file STRING,
    timestamp_ms BIGINT,
    observation_timestamp TIMESTAMP,
    price_usd DOUBLE,
    market_cap_usd DOUBLE,
    total_volume_usd DOUBLE,
    rejection_reason STRING,
    bronze_load_timestamp TIMESTAMP,
    load_timestamp TIMESTAMP NOT NULL,
    run_id STRING
) USING DELTA
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.default.pipeline_execution_logs (
    run_id STRING,
    layer STRING,
    table_name STRING,
    load_type STRING,
    processing_date STRING,
    source_folder STRING,
    start_timestamp TIMESTAMP,
    end_timestamp TIMESTAMP,
    status STRING,
    rows_read BIGINT,
    rows_inserted BIGINT,
    rows_updated BIGINT,
    rows_quarantined BIGINT,
    error_message STRING
) USING DELTA
""")

print("All tables ready.")

In [0]:
# Row count of every table (a quick health check)
for t in ["bronze_coin_market_raw", "bronze_quarantine", "bronze_file_quarantine",
          "silver_market_observation", "silver_quarantine", "pipeline_execution_logs"]:
    print(t, "->", spark.table(f"workspace.default.{t}").count())